In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))  # go up two levels (from numerics to PauliSampling.jl)

  Activating project at `~/Desktop/PauliSampling.jl`


In [2]:
# Load all packages
using PauliSampling, PauliPropagation, Plots, LinearAlgebra, ProgressMeter, Statistics, Printf, Distributions, Random, Optimisers, ReverseDiff, MAT, BenchmarkTools, IterTools, StatsBase, LaTeXStrings, Measures, CSV, DataFrames, JSON3, Dates

In [3]:
using MLDatasets, ImageTransformations, Statistics, LinearAlgebra
using Random, Zygote
using Optim

In [12]:
ENV["DATADEPS_ALWAYS_ACCEPT"] = "true"
# ==============================================================================
# 1. IMPORTS
# ==============================================================================
using PauliSampling, PauliPropagation # Your packages
using Plots, LinearAlgebra, Statistics, Random, Printf
using MLDatasets, ImageTransformations
using Optim
using FiniteDifferences # For black-box gradient estimation

# ==============================================================================
# 2. YOUR EXISTING HELPER FUNCTIONS (Preserved)
# ==============================================================================

# Turn H::Vector{PauliString} into a coefficient vector
function coeffs_from_paulistrings(H)
    return [t.coeff for t in H]
end

# Update coefficients inside H in place
function update_paulistring_coeffs!(H, coeffs)
    @assert length(H) == length(coeffs)
    for i in 1:length(H)
        H[i] = PauliString(H[i].nqubits, H[i].term, coeffs[i])
    end
end

# ==============================================================================
# 3. MNIST DATA PREPARATION
# ==============================================================================

"""
    get_mnist_distribution(digit, n_side)
Fetches MNIST, filters for a specific digit, resizes to (n_side, n_side),
and converts to a probability distribution over bitstrings.
"""
function get_mnist_distribution(digit::Int, n_side::Int)
    println("Loading MNIST data for digit $digit...")
    train_x, train_y = MNIST(split=:train)[:]
    indices = findall(x -> x == digit, train_y)
    images = train_x[:, :, indices]
    
    nq = n_side^2
    bitstring_counts = Dict{String, Int}()
    
    println("Processing $(length(indices)) images into $n_side x $n_side bitstrings...")
    
    for i in 1:size(images, 3)
        img = imresize(images[:, :, i], (n_side, n_side))
        bits = img .> 0.5 # Binarize
        s = join(Int.(bits[:]))
        bitstring_counts[s] = get(bitstring_counts, s, 0) + 1
    end
    
    total = sum(values(bitstring_counts))
    all_bitstrings = [lpad(string(i, base=2), nq, '0') for i in 0:(2^nq - 1)]
    
    probs = zeros(2^nq)
    for (idx, s) in enumerate(all_bitstrings)
        if haskey(bitstring_counts, s)
            probs[idx] = bitstring_counts[s] / total
        end
    end
    
    return all_bitstrings, probs, nq
end

# ==============================================================================
# 4. MMD HELPERS
# ==============================================================================

"""
    sample_mmd_masks(nq, σ, num_samples)
Samples Pauli-Z masks 'a' from the Bernoulli distribution P_σ(a).
"""
function sample_mmd_masks(nq, σ, num_samples)
    p_bernoulli = (1.0 - exp(-1.0 / (2.0 * σ))) / 2.0
    masks = [rand(nq) .< p_bernoulli for _ in 1:num_samples]
    return masks
end

"""
    compute_target_expectations(bitstrings, probs, masks)
Pre-computes <Z_a>_p for the sampled masks.
"""
function compute_target_expectations(bitstrings, probs, masks)
    num_masks = length(masks)
    target_exp = zeros(num_masks)
    binary_data = [parse.(Int, collect(s)) for s in bitstrings]
    
    for k in 1:num_masks
        a = masks[k]
        val = 0.0
        for (idx, x) in enumerate(binary_data)
            if probs[idx] > 0.0
                dot_prod = sum(x .* a)
                parity = iseven(dot_prod) ? 1.0 : -1.0
                val += probs[idx] * parity
            end
        end
        target_exp[k] = val
    end
    return target_exp
end

# ==============================================================================
# 5. MMD OBJECTIVE FUNCTION (Using Your Simulation Code)
# ==============================================================================

"""
    compute_mmd_loss(coeffs, H, target_exp, masks, nq, num_layers, max_weight, min_abs_coeff)
    
1. Updates H with coeffs.
2. Simulates QBM state `rho` using your `makethermalstate`.
3. Extracts probabilities from `rho`.
4. Computes MMD squared.
"""
function compute_mmd_loss(coeffs, H, target_exp, masks, nq, num_layers, max_weight, min_abs_coeff)
    # 1. Update Hamiltonian Coefficients (In-Place)
    update_paulistring_coeffs!(H, coeffs)

    # 2. Simulate QBM State (Using your code)
    circuit, theta = paulistringtocircuit(H)
    rho = makethermalstate(nq, circuit, theta, num_layers;
                           max_weight=max_weight,
                           min_abs_coeff=min_abs_coeff)
    
    # 3. Extract Probabilities (Assumes rho allows diagonal access)
    # Since rho comes from makethermalstate, we assume it's a matrix or supports diag()
    model_probs = real.(diag(rho)) 
    
    # 4. Compute MMD
    num_masks = length(masks)
    loss = 0.0
    
    for k in 1:num_masks
        a = masks[k]
        
        # Calculate <Z_a>_q = sum_s P(s) * (-1)^(s . a)
        model_exp = 0.0
        for (idx, p_s) in enumerate(model_probs)
            state_int = idx - 1
            dot_prod = 0
            for bit_idx in 0:(nq-1)
                if ((state_int >> bit_idx) & 1) == 1 && a[bit_idx+1]
                    dot_prod += 1
                end
            end
            parity = iseven(dot_prod) ? 1.0 : -1.0
            model_exp += p_s * parity
        end
        
        loss += (target_exp[k] - model_exp)^2
    end
    
    return loss / num_masks
end

# ==============================================================================
# 6. TRAINING PIPELINE
# ==============================================================================

function train_qbm_mnist_mmd(digit=0, n_side=3, num_epochs=100)
    # --- 1. Data & Setup ---
    # FIX: Use $(...) for interpolation
    println("--- 1. Preparing Data (Digit $digit, $(n_side)x$(n_side)) ---") 
    bitstrings, target_probs, nq = get_mnist_distribution(digit, n_side)
    
    # MMD Parameters
    σ = nq / 4.0 # Linear scaling for trainability
    num_masks = 100 # Number of Pauli strings to sample
    masks = sample_mmd_masks(nq, σ, num_masks)
    
    # Precompute Target Expectations
    target_exp = compute_target_expectations(bitstrings, target_probs, masks)
    
    # --- 2. Initialize QBM ---
    println("--- 2. Initializing QBM ---")
    rng = MersenneTwister(42)
    
    # Using your parameters() function
    params = parameters(
        :heisenberg_fields, nq;
        init = :randn,
        field_scale = 0.5,
        coupling_scale = 0.3,
        rng = rng
    )
    
    # Initial Hamiltonian
    beta_init = 2.0
    qbm_H_paulis = makehamiltonian(params; connectivity=:nearest, periodic=false) * beta_init
    
    # Simulation Params (Your settings)
    num_layers_qbm = nq * 10
    max_weight_qbm = nq
    min_abs_coeff_qbm = 1e-10
    
    initial_coeffs = coeffs_from_paulistrings(qbm_H_paulis)
    
    # --- 3. Define Objective & Gradient ---
    
    # Closure for the loss function
    function loss_closure(c)
        return compute_mmd_loss(c, qbm_H_paulis, target_exp, masks, 
                                nq, num_layers_qbm, max_weight_qbm, min_abs_coeff_qbm)
    end

    # Gradient function using FiniteDifferences
    function grad_closure!(G, c)
        grads = FiniteDifferences.grad(central_fdm(5, 1), loss_closure, c)[1]
        copy!(G, grads)
    end

    # --- 4. Optimization ---
    println("--- 3. Starting Optimization (LBFGS with FiniteDiff Gradients) ---")
    
    opt = Optim.Options(
        iterations = num_epochs,
        show_trace = true,
        store_trace = true,
        g_tol = 1e-5
    )
    
    result = optimize(
        loss_closure,
        grad_closure!,
        initial_coeffs,
        LBFGS(),
        opt
    )
    
    println("Optimization Finished. Final Loss: $(result.minimum)")
    
    # --- 5. Final Visualization ---
    visualize_final_result(result.minimizer, qbm_H_paulis, n_side, nq, 
                           num_layers_qbm, max_weight_qbm, min_abs_coeff_qbm, bitstrings)
    
    return result
end

function visualize_final_result(coeffs, H, n_side, nq, n_layers, max_w, min_c, bitstrings_all)
    update_paulistring_coeffs!(H, coeffs)
    circuit, theta = paulistringtocircuit(H)
    rho = makethermalstate(nq, circuit, theta, n_layers; max_weight=max_w, min_abs_coeff=min_c)
    model_probs = real.(diag(rho))
    
    # Reconstruct Image
    final_img = zeros(n_side, n_side)
    for idx in 1:2^nq
        state = idx - 1
        p_s = model_probs[idx]
        grid = zeros(n_side, n_side)
        for i in 0:(nq-1)
            val = (state >> i) & 1
            col = div(i, n_side) + 1
            row = (i % n_side) + 1
            grid[row, col] = val
        end
        final_img += p_s .* grid
    end
    
    heatmap(final_img, title="Reconstructed Average Image", color=:greys, aspect_ratio=:equal, yflip=true)
    display(plot(model_probs, title="Learnt Distribution PMF", label="Model"))
end

# ==============================================================================
# 7. RUN
# ==============================================================================
# Train on Digit 0, 3x3 grid (9 qubits)
res = train_qbm_mnist_mmd(0, 3, 50)

--- 1. Preparing Data (Digit 0, 3x3) ---
Loading MNIST data for digit 0...
Processing 5923 images into 3 x 3 bitstrings...
--- 2. Initializing QBM ---
--- 3. Starting Optimization (LBFGS with FiniteDiff Gradients) ---


MethodError: MethodError: no method matching diag(::PauliSum{PauliPropagation.UInt24, Float64})
The function `diag` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  diag(!Matched::BitMatrix)
   @ LinearAlgebra /Applications/Julia-1.11.app/Contents/Resources/julia/share/julia/stdlib/v1.11/LinearAlgebra/src/bitarray.jl:79
  diag(!Matched::FillArrays.Eye)
   @ FillArrays ~/.julia/packages/FillArrays/Ksvco/src/FillArrays.jl:505
  diag(!Matched::FillArrays.RectDiagonal)
   @ FillArrays ~/.julia/packages/FillArrays/Ksvco/src/FillArrays.jl:458
  ...
